# OP GRN Biology Case Study

Single dataset (PBMC multiome, 25k cells, 3 donors).
Analysis: top TFs by out-degree, TF coverage vs PBMC markers, motif DB coverage.


## 1. Imports and Parameters


In [22]:
import os, sys, pickle
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


In [23]:
RESULTS_OP = '/home/jnourisa/projs/ongoing/task_grn_inference/resources/results/op'
OUT_DIR = '/home/jnourisa/projs/ongoing/task_grn_inference/resources/results/experiment/op_network'
os.makedirs(OUT_DIR, exist_ok=True)

sys.path.insert(0, '/home/jnourisa/projs/ongoing/geneRNBI')
sys.path.insert(0, '/home/jnourisa/projs/ongoing')
from src.helper import surrogate_names, palette_methods, colors_blind

TF_ALL_PATH = '/vol/projects/jnourisa/genernbi/resources/grn_benchmark/prior/tf_all.csv'
tf_all = set(pd.read_csv(TF_ALL_PATH).iloc[:, 0].astype(str).str.strip())


In [24]:
METHODS = ['grnboost', 'scenicplus', 'celloracle', 'ppcor', 'granie']
METHOD_COLORS = {m: palette_methods.get(surrogate_names.get(m, m), 'grey') for m in METHODS}
METHOD_COLORS['celloracle'] = '#4CAF50'

PBMC_CORE_TFS = [
    'SPI1', 'IRF8', 'IRF4', 'BATF', 'TBX21', 'GATA3', 'RORC', 'FOXP3',
    'PAX5', 'EBF1', 'TCF7', 'LEF1', 'EOMES', 'IKZF1', 'IKZF2', 'CEBPA',
    'CEBPB', 'MAFB', 'RUNX1', 'RUNX3', 'BACH2', 'ZEB2', 'BCL11B', 'TCF4',
    'ETS1', 'KLF2', 'PRDM1', 'XBP1', 'ZNF683', 'IRF7',
]  # n=30

pbmc_set = set(PBMC_CORE_TFS)


In [25]:
print(f'PBMC core TFs: {len(PBMC_CORE_TFS)}')


PBMC core TFs: 30


In [26]:
import sys
sys.path.insert(0, '/home/jnourisa/projs/ongoing/task_grn_inference')
from src.utils.util import process_links

MAX_N_LINKS = 50000

def load_grn(method):
    path = f"{RESULTS_OP}/op.{method}.{method}.prediction.h5ad"
    if not os.path.exists(path):
        print(f'  [MISSING] {path}')
        return None
    net = ad.read_h5ad(path).uns['prediction']
    par = {'max_n_links': MAX_N_LINKS, 'verbose': 0}
    return process_links(net, par)

_TF_COLOR_PBMC = '#56B4E9'  # known PBMC marker
_TF_COLOR_NONE = '#BDBDBD'  # other

def _tf_color(lbl):
    return _TF_COLOR_PBMC if lbl in pbmc_set else _TF_COLOR_NONE

def _hbar(ax, labels, values):
    bar_colors = [_tf_color(lbl) for lbl in labels]
    y = range(len(labels))
    ax.barh(list(y), values, color=bar_colors, edgecolor='white', linewidth=0.4, alpha=0.85)
    ax.set_yticks(list(y))
    ax.set_yticklabels(labels, fontsize=8)
    ax.invert_yaxis()
    ax.spines[['top', 'right']].set_visible(False)


## 1b. Topological Analysis


In [27]:
rows = []
for method in METHODS:
    grn = load_grn(method)
    if grn is None: continue
    rows.append({
        'Method'  : surrogate_names.get(method, method),
        'Edges'   : len(grn),
        'TFs'     : grn['source'].nunique(),
        'Targets' : grn['target'].nunique(),
    })

df_topo = pd.DataFrame(rows).set_index('Method')
print(df_topo.to_string())


            Edges   TFs  Targets
Method                          
GRNBoost2   50000   925     6991
Scenic+     50000   329    10929
CellOracle  50000   407     5956
PPCOR       50000  1052    13150
GRaNIE      50000    91    11778


## 2. Top TFs by Out-degree


In [28]:
def fig_top_tfs():
    fig, axes = plt.subplots(1, 5, figsize=(20, 5))
    fig.suptitle('Top 20 TFs by Out-degree · OP (PBMC multiome)', fontsize=13, fontweight='bold')
    for i, (ax, method) in enumerate(zip(axes, METHODS)):
        grn = load_grn(method)
        if grn is None:
            ax.text(0.5, 0.5, 'Not available', ha='center', va='center',
                    transform=ax.transAxes, color='grey')
            ax.set_title(surrogate_names.get(method, method), fontsize=10)
            continue
        od = grn.groupby('source')['target'].count().sort_values(ascending=False).head(20)
        _hbar(ax, od.index.tolist(), od.values.tolist())
        ax.set_xlabel('Out-degree (# targets)', fontsize=8)
        ax.set_title(surrogate_names.get(method, method),
                     fontweight='bold', color=METHOD_COLORS[method], fontsize=10)
        n_known = sum(1 for tf in od.index if tf in pbmc_set)
        ax.annotate(f'{n_known}/20 PBMC TFs',
                    xy=(0.97, 0.03), xycoords='axes fraction', ha='right', fontsize=7.5, color='grey')
    legend_handles = [mpatches.Patch(color=_TF_COLOR_PBMC, label='Known PBMC TF')]
    fig.legend(handles=legend_handles, ncol=1, fontsize=9, frameon=False, bbox_to_anchor=(.9, .08))
    plt.tight_layout()
    out = os.path.join(OUT_DIR, 'fig1_top20_tfs_op.png')
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Saved: {out}')


In [29]:
fig_top_tfs()


Saved: /home/jnourisa/projs/ongoing/task_grn_inference/resources/results/experiment/op_network/fig1_top20_tfs_op.png


## 3. TF Coverage Analysis (PBMC markers)


In [30]:
# GWAS analysis not applicable for OP — skipped


In [31]:
# GWAS in-degree — skipped


In [32]:
# GWAS in-degree calls — skipped


## 3b. PBMC Marker TF Coverage per Method

For each GRN method: how many of the 30 PBMC core TFs appear as sources?
Missing TFs are reported with their motif DB status.


In [33]:
method_tfs = {}
for m in METHODS:
    grn = load_grn(m)
    if grn is None:
        method_tfs[m] = set()
        continue
    method_tfs[m] = set(grn['source'].unique())
    print(f'{surrogate_names.get(m,m):12s}: {len(method_tfs[m]):,} unique TFs')

universe_tfs = set().union(*method_tfs.values())
pbmc_in_u = pbmc_set & universe_tfs
print(f'\nUniverse (all methods): {len(universe_tfs):,} unique TFs')
print(f'PBMC core TFs in universe: {len(pbmc_in_u)}/{len(pbmc_set)}')
print(f'Missing from universe: {pbmc_set - universe_tfs}')

method_missing = {}
rows = []
for m in METHODS:
    missing = pbmc_in_u - method_tfs[m]
    method_missing[m] = missing
    rows.append({
        'Method'    : surrogate_names.get(m, m),
        'Present'   : len(pbmc_in_u & method_tfs[m]),
        'Missing'   : len(missing),
        'Total'     : len(pbmc_in_u),
        '% Coverage': f'{100*(len(pbmc_in_u)-len(missing))/len(pbmc_in_u):.1f}%',
        'Missing TFs': ', '.join(sorted(missing)),
    })

df_cov = pd.DataFrame(rows)
print(df_cov[['Method','Present','Missing','Total','% Coverage']].to_string(index=False))
print()
for r in rows:
    print(f"  {r['Method']}: {r['Missing TFs'] if r['Missing TFs'] else 'none missing'}")


GRNBoost2   : 925 unique TFs
Scenic+     : 329 unique TFs
CellOracle  : 407 unique TFs
PPCOR       : 1,052 unique TFs
GRaNIE      : 91 unique TFs

Universe (all methods): 1,134 unique TFs
PBMC core TFs in universe: 30/30
Missing from universe: set()
    Method  Present  Missing  Total % Coverage
 GRNBoost2       30        0     30     100.0%
   Scenic+       20       10     30      66.7%
CellOracle       29        1     30      96.7%
     PPCOR       30        0     30     100.0%
    GRaNIE       13       17     30      43.3%

  GRNBoost2: none missing
  Scenic+: BATF, BCL11B, CEBPA, FOXP3, GATA3, IRF4, IRF8, PRDM1, TBX21, ZNF683
  CellOracle: ZEB2
  PPCOR: none missing
  GRaNIE: BACH2, BATF, BCL11B, EBF1, FOXP3, GATA3, IKZF1, IKZF2, IRF7, PAX5, PRDM1, RORC, RUNX1, TCF4, XBP1, ZEB2, ZNF683


In [34]:
def fig_tf_coverage():
    universe = set().union(*method_tfs.values())
    pbmc_universe = pbmc_set & universe

    fig, ax = plt.subplots(figsize=(5, 2.5))
    method_labels = [surrogate_names.get(m, m) for m in METHODS]

    for i, m in enumerate(METHODS):
        cov   = len(pbmc_universe & method_tfs[m])
        total = len(pbmc_universe)
        pct   = 100 * cov / total if total > 0 else 0
        ax.barh(i, pct, height=0.5, color=_TF_COLOR_PBMC, alpha=0.85,
                edgecolor='white', linewidth=0.5)
        ax.text(pct + 1, i, f'{cov}/{total}', va='center', ha='left', fontsize=9)

    ax.set_yticks(range(len(METHODS)))
    ax.set_yticklabels(method_labels)
    ax.set_xlabel('Coverage of PBMC core TFs (%)', fontsize=9)
    ax.set_xlim(0, 130)
    ax.spines[['top', 'right']].set_visible(False)
    ax.invert_yaxis()
    plt.tight_layout()
    out = os.path.join(OUT_DIR, 'fig2_tf_coverage_op.png')
    plt.savefig(out, dpi=150, bbox_inches='tight', transparent=True)
    plt.close()
    print(f'Saved: {out}')

fig_tf_coverage()


Saved: /home/jnourisa/projs/ongoing/task_grn_inference/resources/results/experiment/op_network/fig2_tf_coverage_op.png


In [35]:
# GWAS TF summary — skipped for OP


## 3c. Motif Database Coverage of Missing Annotated TFs

For each method, we ask: among the PBMC core TFs missed by that method, is the absence explained
by the TF having no motif in the underlying database, or does the TF have a motif but get filtered
out by the method's downstream enrichment thresholds?

- **CellOracle** uses [JASPAR2022](https://jaspar.elixir.no/) motifs (`JASPAR2022-hg38.bed.gz`).
- **SCENIC+** uses the aeretslab cisTarget v10 motif database (`motifs-v10-nr.hgnc-m0.00001-o0.0.tbl`).
- **GRaNIE** uses [HOCOMOCO v12](https://hocomoco12.autosome.org/) in-vivo motifs (`PWMScan_HOCOMOCOv12_H12INVIVO`).

In [36]:
import gzip, os

JASPAR_BED       = '/home/jnourisa/projs/ongoing/task_grn_inference/resources/supp_data/databases/scglue/JASPAR2022-hg38.bed.gz'
MOTIF_SCENIC_TBL = '/home/jnourisa/projs/ongoing/task_grn_inference/resources/supp_data/databases/scenicplus/motifs-v10-nr.hgnc-m0.00001-o0.0.tbl'
HOCOMOCO_DIR     = '/home/jnourisa/projs/ongoing/task_grn_inference/resources/supp_data/databases/granie/H12INVIVO'

# CellOracle: TF names from JASPAR2022 bed (col 4)
jaspar_tfs = set()
with gzip.open(JASPAR_BED, 'rt') as fh:
    for line in fh:
        parts = line.strip().split('\t')
        if len(parts) >= 4:
            jaspar_tfs.add(parts[3].upper())

# SCENIC+: gene_name column in tbl
sp_df = pd.read_csv(MOTIF_SCENIC_TBL, sep='\t', comment='#',
                    header=None, low_memory=False,
                    names=open(MOTIF_SCENIC_TBL).readline().strip().split('\t'))
sp_motif_tfs = set(sp_df['gene_name'].dropna().str.strip().str.upper())

# GRaNIE: TF names from HOCOMOCO v12 file names (part before first dot)
hocomoco_tfs = set(f.split('.')[0].upper() for f in os.listdir(HOCOMOCO_DIR))

print(f'CellOracle motif TFs (JASPAR2022)  : {len(jaspar_tfs):,}')
print(f'SCENIC+    motif TFs (cisTarget v10): {len(sp_motif_tfs):,}')
print(f'GRaNIE     motif TFs (HOCOMOCO v12) : {len(hocomoco_tfs):,}')

method_motif_db = {
    'celloracle': jaspar_tfs,
    'scenicplus' : sp_motif_tfs,
    'granie'     : hocomoco_tfs,
}

print()
for m, motif_db in method_motif_db.items():
    missing = method_missing.get(m, set())
    no_motif  = {tf for tf in missing if tf not in motif_db}
    has_motif = missing - no_motif
    name = surrogate_names.get(m, m)
    print(f'── {name} ──────────────────')
    print(f'  Missing PBMC TFs    : {len(missing)}')
    print(f'  Absent from motif DB: {len(no_motif)} → {sorted(no_motif)}')
    print(f'  Have motif, filtered: {len(has_motif)} → {sorted(has_motif)}')
    print()

CellOracle motif TFs (JASPAR2022)  : 634
SCENIC+    motif TFs (cisTarget v10): 1,605
GRaNIE     motif TFs (HOCOMOCO v12) : 951

── CellOracle ──────────────────
  Missing PBMC TFs    : 1
  Absent from motif DB: 1 → ['ZEB2']
  Have motif, filtered: 0 → []

── Scenic+ ──────────────────
  Missing PBMC TFs    : 10
  Absent from motif DB: 0 → []
  Have motif, filtered: 10 → ['BATF', 'BCL11B', 'CEBPA', 'FOXP3', 'GATA3', 'IRF4', 'IRF8', 'PRDM1', 'TBX21', 'ZNF683']

── GRaNIE ──────────────────
  Missing PBMC TFs    : 17
  Absent from motif DB: 7 → ['BCL11B', 'EBF1', 'IKZF1', 'IKZF2', 'RORC', 'TCF4', 'ZNF683']
  Have motif, filtered: 10 → ['BACH2', 'BATF', 'FOXP3', 'GATA3', 'IRF7', 'PAX5', 'PRDM1', 'RUNX1', 'XBP1', 'ZEB2']

